# GeoLifeCLEF 2025 — CNN ResNet18 sur images satellite
**Approche** : Images satellite 64×64 RGB+NIR → ResNet18 pré-entraîné → prédiction multi-label

Baseline précédente : **0.16395** → Objectif : **0.25+**

Activer GPU T4 x2 : **Session options > Accelerator > GPU T4 x2**

In [ ]:
!pip install timm -q

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import rasterio
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')

DATA    = Path('/kaggle/input/competitions/le-challenge-deep-learning-miashs-2026')
PATCHES = DATA / 'SatelitePatches'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPUs disponibles: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── 1. Chargement des labels ─────────────────────────────────────────────────
pa_train = pd.read_csv(DATA / 'GLC25_PA_metadata_train.csv')
pa_test  = pd.read_csv(DATA / 'GLC25_PA_metadata_test.csv')

train_labels = (
    pa_train.dropna(subset=['speciesId'])
    .groupby('surveyId')['speciesId']
    .apply(lambda x: list(x.astype(int).unique()))
    .reset_index()
    .rename(columns={'speciesId': 'species_list'})
)

all_species = sorted(pa_train['speciesId'].dropna().astype(int).unique())
mlb = MultiLabelBinarizer(classes=all_species)
mlb.fit(train_labels['species_list'])
NUM_CLASSES = len(all_species)

print(f'Train: {len(train_labels)} surveys | Classes: {NUM_CLASSES}')
print(f'Test: {len(pa_test["surveyId"].unique())} surveys')

In [ ]:
# ─── 2. Fonction pour charger une image satellite ─────────────────────────────
def get_patch_path(survey_id, patches_dir):
    """Construit le chemin vers le fichier TIFF d'un surveyId"""
    s = str(survey_id)
    if len(s) >= 4:
        return patches_dir / s[-2:] / s[-4:-2] / f'{survey_id}.tiff'
    else:
        return patches_dir / s / f'{survey_id}.tiff'

def load_patch(survey_id, patches_dir):
    """Charge une image satellite 4 bandes (RGB+NIR) de 64x64"""
    path = get_patch_path(survey_id, patches_dir)
    try:
        with rasterio.open(path) as src:
            img = src.read()  # shape: (4, 64, 64)
        img = img.astype(np.float32)
        # Normaliser chaque bande
        for b in range(img.shape[0]):
            band = img[b]
            mn, mx = band.min(), band.max()
            if mx > mn:
                img[b] = (band - mn) / (mx - mn)
        return img
    except Exception:
        return np.zeros((4, 64, 64), dtype=np.float32)

# Test rapide
sid = train_labels['surveyId'].iloc[0]
img = load_patch(sid, PATCHES)
print(f'Image shape: {img.shape} | min: {img.min():.3f} | max: {img.max():.3f}')

In [ ]:
# ─── 3. Dataset PyTorch ───────────────────────────────────────────────────────
class SatelliteDataset(Dataset):
    def __init__(self, survey_ids, labels_df, patches_dir, mlb, is_train=True):
        self.survey_ids  = survey_ids
        self.patches_dir = patches_dir
        self.is_train    = is_train
        self.mlb         = mlb

        # Créer un dict surveyId → species_list
        if labels_df is not None:
            self.label_dict = dict(zip(labels_df['surveyId'], labels_df['species_list']))
        else:
            self.label_dict = {}

    def __len__(self):
        return len(self.survey_ids)

    def __getitem__(self, idx):
        sid = self.survey_ids[idx]
        img = load_patch(sid, self.patches_dir)  # (4, 64, 64)
        img = torch.tensor(img)

        # Augmentation légère en train
        if self.is_train:
            if torch.rand(1) > 0.5:
                img = torch.flip(img, dims=[2])  # flip horizontal
            if torch.rand(1) > 0.5:
                img = torch.flip(img, dims=[1])  # flip vertical

        if sid in self.label_dict:
            label = self.mlb.transform([self.label_dict[sid]])[0].astype(np.float32)
        else:
            label = np.zeros(len(self.mlb.classes_), dtype=np.float32)

        return img, torch.tensor(label)

# Split train/val 80/20
from sklearn.model_selection import train_test_split
all_ids = train_labels['surveyId'].values
tr_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
test_ids = pa_test['surveyId'].unique()

train_ds = SatelliteDataset(tr_ids,  train_labels, PATCHES, mlb, is_train=True)
val_ds   = SatelliteDataset(val_ids, train_labels, PATCHES, mlb, is_train=False)
test_ds  = SatelliteDataset(test_ids, None,        PATCHES, mlb, is_train=False)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# ─── 4. Modèle ResNet18 adapté 4 canaux ───────────────────────────────────────
class ResNet18_4ch(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Charger ResNet18 pré-entraîné
        self.base = timm.create_model('resnet18', pretrained=True, num_classes=0)

        # Adapter le premier conv pour 4 canaux (RGB + NIR)
        # On reprend les poids RGB et on initialise NIR = moyenne RGB
        old_conv = self.base.conv1
        new_conv = nn.Conv2d(4, old_conv.out_channels,
                             kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride,
                             padding=old_conv.padding,
                             bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3]  = old_conv.weight.mean(dim=1)
        self.base.conv1 = new_conv

        # Tête de classification
        in_features = self.base.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        features = self.base(x)
        return self.classifier(features)

model = ResNet18_4ch(NUM_CLASSES)

# Multi-GPU si disponible
if torch.cuda.device_count() > 1:
    print(f'Utilisation de {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)

model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Paramètres: {total_params:,}')

In [ ]:
# ─── 5. Entraînement ──────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item()
            probs = torch.sigmoid(outputs).cpu().numpy()
            val_preds.append(probs)
            val_true.append(labels.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_true  = np.vstack(val_true)

    # F-score local
    THRESHOLD, TOP_K = 0.2, 10
    f_scores = []
    for i in range(len(val_preds)):
        above = np.where(val_preds[i] >= THRESHOLD)[0]
        if len(above) < TOP_K:
            above = np.argsort(val_preds[i])[::-1][:TOP_K]
        pred_set = set(above)
        true_set = set(np.where(val_true[i] == 1)[0])
        tp = len(pred_set & true_set)
        if tp == 0:
            f_scores.append(0.0)
            continue
        p = tp / len(pred_set)
        r = tp / len(true_set)
        f_scores.append(2 * p * r / (p + r))

    avg_f = np.mean(f_scores)
    scheduler.step()

    tl = train_loss / len(train_dl)
    vl = val_loss   / len(val_dl)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | F-score: {avg_f:.4f}')

    # Sauvegarder le meilleur modèle
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        print(f'  Meilleur modèle sauvegardé (val_loss={vl:.4f})')

In [ ]:
# ─── 6. Prédiction sur le test ────────────────────────────────────────────────
# Charger le meilleur modèle
model.load_state_dict(torch.load('/kaggle/working/best_model.pt'))
model.eval()

test_probs_all = []
with torch.no_grad():
    for imgs, _ in test_dl:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.sigmoid(outputs).cpu().numpy()
        test_probs_all.append(probs)

test_probs_all = np.vstack(test_probs_all)
print(f'Prédictions test: {test_probs_all.shape}')

In [ ]:
# ─── 7. Génération soumission ─────────────────────────────────────────────────
THRESHOLD = 0.2
TOP_K = 10

predictions = []
for i in range(len(test_probs_all)):
    probs = test_probs_all[i]
    above = np.where(probs >= THRESHOLD)[0]
    if len(above) < TOP_K:
        top_k = np.argsort(probs)[::-1][:TOP_K]
        pred_idx = np.union1d(above, top_k)
    else:
        pred_idx = above
    pred_species = sorted([int(mlb.classes_[j]) for j in pred_idx])
    predictions.append(' '.join(map(str, pred_species)))

submission = pd.DataFrame({
    'surveyId': test_ids,
    'predictions': predictions
})
submission.to_csv('/kaggle/working/submission_cnn.csv', index=False)
print(f'Soumission CNN sauvegardée! {len(submission)} surveys')
submission.head()